# 7.29 — Video understanding & action recognition

Video understanding is image understanding with time made explicit: evidence must be pooled, ordered, or convolved across frames before an action can be named.

Action recognition asks what happened, not merely what appears in one frame. Frame encoders extract spatial features from each image; temporal pooling, differences, and 3D-style filters combine those features when motion direction matters. Save a copy to Drive to edit.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(729)
np.set_printoptions(precision=4, suppress=True)

## Build the clip classifier once

The lesson's baseline averages frame features:

$$ar h=rac{1}{T}\sum_{t=1}^{T}h_t,\qquad p(y\mid x_{1:T})=\operatorname{softmax}(War h+b)$$

Mean pooling is a useful first descriptor, but it forgets the order of the frames.

In [ ]:
def softmax(logits):
    shifted = logits - np.max(logits)
    exp = np.exp(shifted)
    return exp / exp.sum()


def classify_clip(frame_features):
    frame_features = np.asarray(frame_features, dtype=float)
    pooled = frame_features.mean(axis=0)
    motion = np.diff(frame_features[:, 0]).mean()
    weights = np.array([[1.0, 0.0], [0.0, 1.0]])
    logits = weights @ pooled
    probs = softmax(logits)
    return pooled, motion, logits, probs

features = np.array([[1, 0], [2, 1], [3, 1]], dtype=float)
reversed_features = features[::-1]
pooled, motion, logits, probs = classify_clip(features)
pooled_rev, motion_rev, logits_rev, probs_rev = classify_clip(reversed_features)
print("pooled", pooled)
print("reversed pooled", pooled_rev)
print("motion signs", motion, motion_rev)
print("run probability", probs[0])
assert np.allclose(pooled, [2.0, 2.0 / 3.0])
assert np.allclose(pooled_rev, pooled)
assert motion == 1.0
assert motion_rev == -1.0
assert np.isclose(probs[0], 0.791391472673955, atol=1e-12)

A tiny 3D temporal filter sees local motion directly. With kernel $[1,-1]$, values $1,3$ produce $-2$ and the reversed order produces $2$.

In [ ]:
def temporal_filter_response(values):
    values = np.asarray(values, dtype=float)
    kernel = np.array([1.0, -1.0])
    return float(values[:2] @ kernel)

forward_response = temporal_filter_response([1, 3])
reverse_response = temporal_filter_response([3, 1])
print("3D-style temporal responses", forward_response, reverse_response)
assert forward_response == -2.0
assert reverse_response == 2.0

## Synthetic clip ladder

Each rung contains clips of a bright dot moving left-to-right or right-to-left. Difficulty rises by adding more frames, distractors, noise, partial occlusion, and a harder background. The same temporal method is used on every rung.

In [ ]:
def make_clip(direction, size, frames, noise, distractors, occlusion, seed):
    local = np.random.default_rng(seed)
    clip = local.normal(0.0, noise, size=(frames, size, size))
    row = size // 2
    start = 1
    stop = size - 2
    xs = np.linspace(start, stop, frames)
    if direction < 0:
        xs = xs[::-1]
    for t, x_float in enumerate(xs):
        x = int(round(x_float))
        if local.random() > occlusion:
            clip[t, row, x] += 1.0
            if row + 1 < size:
                clip[t, row + 1, x] += 0.5
        for _ in range(distractors):
            yy = local.integers(0, size)
            xx = local.integers(0, size)
            clip[t, yy, xx] += local.uniform(0.2, 0.8)
    clip = np.clip(clip, 0.0, 1.0)
    return clip


def build_clip_rung(name, size, frames, n_each, noise, distractors, occlusion, seed):
    clips = []
    labels = []
    for i in range(n_each):
        clips.append(make_clip(1, size, frames, noise, distractors, occlusion, seed + i))
        labels.append(1)
        clips.append(make_clip(-1, size, frames, noise, distractors, occlusion, seed + 1000 + i))
        labels.append(0)
    return name, np.array(clips), np.array(labels)


def clip_feature(clip):
    frames, height, width = clip.shape
    xs = np.arange(width)
    masses = clip.sum(axis=(1, 2)) + 1e-9
    x_positions = (clip.sum(axis=1) @ xs) / masses
    pooled_intensity = clip.mean()
    motion = np.diff(x_positions).mean()
    return np.array([pooled_intensity, motion])


def predict_clip(clip):
    feat = clip_feature(clip)
    return int(feat[1] > 0.0)


def accuracy(clips, labels):
    preds = np.array([predict_clip(clip) for clip in clips])
    return float((preds == labels).mean())

rungs = [
    build_clip_rung("D1 tiny clean motion", 6, 3, 18, 0.00, 0, 0.00, 10),
    build_clip_rung("D2 longer clips", 8, 5, 24, 0.03, 1, 0.00, 20),
    build_clip_rung("D3 noisy distractors", 10, 6, 30, 0.08, 2, 0.05, 30),
    build_clip_rung("D4 occluded action", 12, 8, 36, 0.12, 4, 0.18, 40),
    build_clip_rung("D5 cluttered sparse action", 14, 8, 42, 0.18, 7, 0.30, 50),
]
for name, clips, labels in rungs:
    print(f"{name:28s} clips={clips.shape} classes={sorted(set(labels.tolist()))}")

In [ ]:
metrics = []
for name, clips, labels in rungs:
    acc = accuracy(clips, labels)
    metrics.append(acc)
    print(f"{name:28s} accuracy={acc:.3f}")

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(14, 5))
for idx, (name, clips, labels) in enumerate(rungs):
    clip = clips[0]
    axes[0, idx].imshow(clip.max(axis=0), cmap="magma", vmin=0, vmax=1)
    axes[0, idx].set_title(name.split()[0])
    axes[0, idx].axis("off")
axes[1, 0].plot(range(1, 6), metrics, marker="o")
axes[1, 0].set_ylim(0, 1.05)
axes[1, 0].set_xlabel("rung")
axes[1, 0].set_ylabel("accuracy")
axes[1, 0].set_title("Temporal method vs difficulty")
for idx in range(1, 5):
    axes[1, idx].axis("off")
fig.tight_layout()

## Pitfall on D5: averaging away the action

The lesson warns that forward and reversed clips can have the same mean feature. On the hardest rung, a mean-intensity-only baseline cannot know direction; adding the temporal difference restores the sign cue.

In [ ]:
name, clips, labels = rungs[-1]
mean_scores = np.array([clip.mean() for clip in clips])
mean_threshold = np.median(mean_scores)
mean_preds = (mean_scores > mean_threshold).astype(int)
fixed_preds = np.array([predict_clip(clip) for clip in clips])
mean_acc = float((mean_preds == labels).mean())
fixed_acc = float((fixed_preds == labels).mean())
print("mean-only accuracy", round(mean_acc, 3))
print("temporal-difference accuracy", round(fixed_acc, 3))
assert fixed_acc >= mean_acc

## Evaluate it + practice

- Track action accuracy and compare with a no-skill baseline that guesses the majority class.
- Sanity check: overfit D1 or inspect one clip's center-of-mass trace.
- Ablation: remove the temporal difference; forward and reverse clips collapse.
- Failure signals: high accuracy from backgrounds, strong drops under occlusion, or sensitivity to frame sampling.

Practice: change the occlusion rate in D5 and rerun the metric.

In [ ]:
# Your experiment here

Practice: replace the dot with two moving dots and ask whether the feature still captures direction.

In [ ]:
# Your experiment here

Practice: compute a two-frame temporal filter response for every adjacent frame pair.

In [ ]:
# Your experiment here